In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import uproot
import glob

In [ ]:
def applyCut(inputDataframe, cut):
    dataframe = inputDataframe
    cutDataframe = dataframe.query(cut)
    return cutDataframe

In [ ]:
def getData(observables, filename):
    return (uproot.open(filename)["MatchedTree"]).arrays(observables, library='pd')

In [ ]:
# add new columns 'a' and 'loga' to dataframe
def addNewBranches(inputDataframe):
    df_temp = inputDataframe.copy()
    df_temp.eval('a=(m*m-mg*mg)/(pt*pt)', inplace=True)
    df_temp['a'] = df_temp['a'].mask(df_temp['a'] < 1e-4, 7.9e-5)
    df_temp.eval('loga=log10(a)', inplace=True)
    return df_temp

In [ ]:
def get_eff(ob, binedges, truth_cut, file=None):
    if file is None:
        file = inputFiles[0]
    df = applyCut(addNewBranches(getData(observables+observabless, file)), truth_cut)
    match_miss, _ = np.histogram(df[ob], bins=binedges, weights=df['w'])
    match, _ = np.histogram(df[(df.pts>15)&(df.w<5e-9)][ob], bins=binedges, weights=df[(df.pts>15)&(df.w<5e-9)]['w'])
    return match/(1e-50+match_miss)

In [ ]:
def calc_hist_nonuniform(vals, bins, weights, density=False):
    if weights is None:
        weights = np.ones(vals.shape)
    hist, bins = np.histogram(vals, bins=bins, weights=weights)
    digits = np.digitize(vals, bins)
    errs = np.asarray([np.linalg.norm(weights[digits==i]) for i in range(1, len(bins))])
    for i in range(1, len(bins)):
        hist[i-1] /= (bins[i] - bins[i-1])
        errs[i-1] /= (bins[i] - bins[i-1])       
    if density:
        hist /= weights.sum()
        errs /= weights.sum()   
    return hist, errs, bins

# Read in prior

In [ ]:
observables = ['pt','q','m','rg','zg','mg','w']
observabless = ['pts']#,'qs','ms','rgs','zgs','mgs']
inputFiles = ['/nfs/roberts/home/ys668/output/run12/all050124_rc_notrueptcons.root',\
             '/nfs/roberts/home/ys668/output/run12/all050124_rc_eff096_notrueptcons.root',\
             '/nfs/roberts/home/ys668/output/run12/all050124_rc_tow104_notrueptcons.root',\
             '/nfs/roberts/home/ys668/output/run12/all050124_rc_hadcorr05_notrueptcons.root']
df1 = applyCut(getData(observables+observabless, inputFiles[0]), 'pt>0 and pts>0 and w<5e-9')
df1 = addNewBranches(df1)

In [ ]:
df1_eff096 = applyCut(getData(observables+observabless, inputFiles[1]), 'pt>0 and pts>0 and w<5e-9')
df1_eff096 = addNewBranches(df1_eff096)

In [ ]:
df1_tow104 = applyCut(getData(observables+observabless, inputFiles[2]), 'pt>0 and pts>0 and w<5e-9')
df1_tow104 = addNewBranches(df1_tow104)

In [ ]:
df1_hadcorr05 = applyCut(getData(observables+observabless, inputFiles[3]), 'pt>0 and pts>0 and w<5e-9')
df1_hadcorr05 = addNewBranches(df1_hadcorr05)

In [ ]:
# w_hw = pd.read_csv('/nfs/roberts/home/ys668/output/run12/herwig_20235003_notrueptcons.csv')['relw_new'] # file doesn't exist yet
# df1['w_hw'] = np.asarray(w_hw)

In [ ]:
# w_py8 = pd.read_csv('/nfs/roberts/home/ys668/output/run12/pythia_20235003_notrueptcons.csv')['relw_new'] # file doesn't exist yet
# df1['w_py8'] = np.asarray(w_py8)

# Read in unfolded

In [ ]:
for it in range(1,9):
    filelist = glob.glob("/home/ys668/project_pi_hlc7/ys668/run12/csvs/final_0628/20235003_data_batch50000_notrueptcons/it"+str(it)+'_*')
    if (it == 1):
        print("Each iteration has", len(filelist), "trials.")
    for file in filelist:
        df_temp = pd.read_csv(file, usecols=['weight'], dtype={'weight': np.float32})
        seed_id = file.replace('.csv','').split("seed",1)[1]
        column_name = "w_it" + str(it) + "_seed" + seed_id
        df1[column_name] = np.asarray(df_temp['weight'])

In [ ]:
for it in range(2,3):
    filelist = glob.glob("/nfs/roberts/pi/pi_lbh35/youqi/csvs/20235003_stat_notrueptcons/it"+str(it)+'_*')
    if (it == 2):
        print("Each iteration has", len(filelist), "trials.")
    for file in filelist:
        df_temp = pd.read_csv(file, usecols=['weight'], dtype={'weight': np.float32})
        seed_id = file.replace('.csv','').split("statistic",1)[1]
        column_name = "w_it" + str(it) + "_bsmc" + seed_id
        df1[column_name] = np.asarray(df_temp['weight'])

In [ ]:
for it in range(2,3):
    filelist = glob.glob("/nfs/roberts/pi/pi_lbh35/youqi/csvs/20235003_datastat_notrueptcons/it"+str(it)+'_*')
    if (it == 2):
        print("Each iteration has", len(filelist), "trials.")
    for file in filelist:
        df_temp = pd.read_csv(file, usecols=['weight'], dtype={'weight': np.float32})
        seed_id = file.replace('.csv','').split("statistic",1)[1]
        column_name = "w_it" + str(it) + "_bsdata" + seed_id
        df1[column_name] = np.asarray(df_temp['weight'])

In [ ]:
for it in range(2,3):
    filelist = glob.glob("/nfs/roberts/pi/pi_lbh35/youqi/csvs/20235003_eff096_notrueptcons/it"+str(it)+'_*')
    if (it == 2):
        print("Each iteration has", len(filelist), "trials.")
    for file in filelist:
        df_temp = pd.read_csv(file, usecols=['weight'], dtype={'weight': np.float32})
        seed_id = file.replace('.csv','').split("seed",1)[1]
        column_name = "w_it" + str(it) + "_eff096seed" + seed_id
        df1_eff096[column_name] = np.asarray(df_temp['weight'])

In [ ]:
for it in range(2,3):
    filelist = glob.glob("/nfs/roberts/pi/pi_lbh35/youqi/csvs/20235003_tow104_notrueptcons/it"+str(it)+'_*')
    if (it == 2):
        print("Each iteration has", len(filelist), "trials.")
    for file in filelist:
        df_temp = pd.read_csv(file, usecols=['weight'], dtype={'weight': np.float32})
        seed_id = file.replace('.csv','').split("seed",1)[1]
        column_name = "w_it" + str(it) + "_tow104seed" + seed_id
        df1_tow104[column_name] = np.asarray(df_temp['weight'])

In [ ]:
for it in range(2,3):
    filelist = glob.glob("/home/ys668/project_pi_hlc7/ys668/run12/csvs/final_0628/20235003_hadcorr05_notrueptcons/it"+str(it)+'_*')
    if (it == 2):
        print("Each iteration has", len(filelist), "trials.")
    for file in filelist:
        df_temp = pd.read_csv(file, usecols=['weight'], dtype={'weight': np.float32})
        seed_id = file.replace('.csv','').split("seed",1)[1]
        column_name = "w_it" + str(it) + "_hadcorr05seed" + seed_id
        df1_hadcorr05[column_name] = np.asarray(df_temp['weight'])

In [ ]:
# for it in range(4,5):
#     filelist = glob.glob("/gpfs/gibbs/project/havener/ys668/run12/20235003_herwig_prior_notrueptcons/it"+str(it)+'_*') # files don't exist yet
#     if (it == 1):
#         print("Each iteration has", len(filelist), "trials.")
#     for file in filelist:
#         df_temp = pd.read_csv(file, low_memory=False)
#         seed_id = file.replace('.csv','').split("seed",1)[1]
#         column_name = "w_it" + str(it) + "_hwseed" + seed_id
#         df1[column_name] = np.asarray(df_temp['weight'])

In [ ]:
# for it in range(4,5):
#     filelist = glob.glob("/gpfs/gibbs/project/havener/ys668/run12/20235003_pythia8_prior/it"+str(it)+'_*') # files don't exist yet
#     if (it == 1):
#         print("Each iteration has", len(filelist), "trials.")
#     for file in filelist:
#         df_temp = pd.read_csv(file, low_memory=False)
#         seed_id = file.replace('.csv','').split("seed",1)[1]
#         column_name = "w_it" + str(it) + "_py8seed" + seed_id
#         df1[column_name] = np.asarray(df_temp['weight'])

In [ ]:
df1 # new columns are results (weights) from different iterations, different seeds (trials), and different bootstrap trials

# Obtain unfolded distribution and do efficiency correction

In [ ]:
def get_seed_unc(df, it, ob, binedges, seed_string="_seed"):
    ys = []
    rels_stat_err = []
    diff_ys_y = []
    for column_name in df.columns:
        if column_name.startswith("w_it" + str(it) + seed_string):
            ys.append(np.histogram(df[ob], bins=binedges, weights=df[column_name], density=True)[0])
            rels_stat_err.append(np.sqrt(np.histogram(df[ob], bins=binedges, weights=df[column_name]**2, density=False)[0])/\
                                 np.histogram(df[ob], bins=binedges, weights=df[column_name], density=False)[0])
    y = np.median(ys, axis=0)
    y_err = 1.253 * np.std(ys, axis=0) / np.sqrt(len(ys))
    print("relative uncertainty from NN:", y_err/y)
    
    for i in range(len(ys)):
        diff_ys_y.append(abs(ys[i] - y))
    diff_ys_y = np.array(diff_ys_y)   
    y_stat_err = []
    for i in range(len(binedges)-1):
        trial_index = np.where(diff_ys_y[:,i] == min(diff_ys_y[:,i]))[0][0] # for bin i, which seed gives the y closest to the median?
        y_stat_err.append(rels_stat_err[trial_index][i]*y[i])
    print("relative uncertainty from stat:", y_stat_err/y)
    
    return y, y_err, y_stat_err # median over all seeds, uncertainty from different seeds, regular statistical uncertainty

In [ ]:
def get_mc_bs_unc(df, it, ob, binedges):  
    ys = []
    for column_name in df.columns:
        if column_name.startswith("w_it" + str(it) + "_bsmc"):
            ys.append(np.histogram(df[ob], bins=binedges, weights=df[column_name], density=True)[0])
    y = np.average(ys, axis=0)
    y_err = np.std(ys, axis=0)
    print("relative uncertainty from MC bootstrap:", y_err/y)
    return y, y_err

In [ ]:
def get_data_bs_unc(df, it, ob, binedges):  
    ys = []
    for column_name in df.columns:
        if column_name.startswith("w_it" + str(it) + "_bsdata"):
            ys.append(np.histogram(df[ob], bins=binedges, weights=df[column_name], density=True)[0])
    y = np.average(ys, axis=0)
    y_err = np.std(ys, axis=0)
    print("relative uncertainty from data bootstrap:", y_err/y)
    return y, y_err

In [ ]:
def process_ob_nom(ob_index):
    
    print("observable:", obs[ob_index])
    ob = obs[ob_index]
    binedges = binedgess[ob_index]
    cut = cuts[ob_index]
    
    x = []
    for i in range(0, len(binedges)-1):
        midpoint = 0.5 * (binedges[i] + binedges[i+1])
        x.append(midpoint)
    
    df1_cut = applyCut(df1, cut)
    eff = get_eff(ob, binedges, cut)
    
    y_arr = []
    err_arr = []
    
    for it in range(1,9):
        print("iteration", it)
        y_seed, err_seed, err_stat = get_seed_unc(df1_cut, it, ob, binedges)
        if (it == 2): # nominal number of iterations
            y_mc_bs, err_mc_bs = get_mc_bs_unc(df1_cut, it, ob, binedges)
            y_data_bs, err_data_bs = get_data_bs_unc(df1_cut, it, ob, binedges)
        else:
            y_mc_bs = np.ones(len(y_seed))
            err_mc_bs = np.zeros(len(y_seed))
            y_data_bs = np.ones(len(y_seed))
            err_data_bs = np.zeros(len(y_seed))
        rel_err = np.sqrt( (err_stat/y_seed)**2 + (err_seed/y_seed)**2 + (err_mc_bs/y_mc_bs)**2 + (err_data_bs/y_data_bs)**2 )
        print('total (stat + NN + MC bootstrap + data bootstrap):', rel_err)
        err = y_seed * rel_err
        
        # scale by efficiency
        y_scaled = y_seed / (1e-50 + eff)
        err_scaled = err / (1e-50 + eff)
        
        # normalize
        bin_widths = np.diff(binedges)
        y_arr.append(y_scaled / sum(y_scaled) / bin_widths)
        err_arr.append(err_scaled / sum(y_scaled) / bin_widths)

    return x, y_arr, err_arr

In [ ]:
def process_ob_hw(ob_index):
    
    print("observable:", obs[ob_index])
    ob = obs[ob_index]
    binedges = binedgess[ob_index]
    cut = cuts[ob_index]
    
    x = []
    for i in range(0, len(binedges)-1):
        midpoint = 0.5 * (binedges[i] + binedges[i+1])
        x.append(midpoint)
    
    df1_cut = applyCut(df1, cut)
    eff = get_eff(ob, binedges, cut)
    
    y_arr = []
    err_arr = []
    
    for it in range(4,5):
        print("iteration", it)
        y_seed, err_seed, err_stat = get_seed_unc(df1_cut, it, ob, binedges, seed_string="_hwseed")
        rel_err = np.sqrt( (err_stat/y_seed)**2 + (err_seed/y_seed)**2 )
        print('total (stat + NN):', rel_err)
        err = y_seed * rel_err
        
        # scale by efficiency
        y_scaled = y_seed / (1e-50 + eff)
        err_scaled = err / (1e-50 + eff)
        
        # normalize
        bin_widths = np.diff(binedges)
        y_arr.append(y_scaled / sum(y_scaled) / bin_widths)
        err_arr.append(err_scaled / sum(y_scaled) / bin_widths)
    
    return x, y_arr, err_arr


In [ ]:
def process_ob_py8(ob_index):
    
    print("observable:", obs[ob_index])
    ob = obs[ob_index]
    binedges = binedgess[ob_index]
    cut = cuts[ob_index]
    
    x = []
    for i in range(0, len(binedges)-1):
        midpoint = 0.5 * (binedges[i] + binedges[i+1])
        x.append(midpoint)
    
    df1_cut = applyCut(df1, cut)
    eff = get_eff(ob, binedges, cut)
    
    y_arr = []
    err_arr = []
    
    for it in range(4,5):
        print("iteration", it)
        y_seed, err_seed, err_stat = get_seed_unc(df1_cut, it, ob, binedges, seed_string="_py8seed")
        rel_err = np.sqrt( (err_stat/y_seed)**2 + (err_seed/y_seed)**2 )
        print('total (stat + NN):', rel_err)
        err = y_seed * rel_err
        
        # scale by efficiency
        y_scaled = y_seed / (1e-50 + eff)
        err_scaled = err / (1e-50 + eff)
        
        # normalize
        bin_widths = np.diff(binedges)
        y_arr.append(y_scaled / sum(y_scaled) / bin_widths)
        err_arr.append(err_scaled / sum(y_scaled) / bin_widths)
    
    return x, y_arr, err_arr


In [ ]:
def process_ob_detector(ob_index, detector_variation, dataframe):
    
    print("observable:", obs[ob_index])
    ob = obs[ob_index]
    binedges = binedgess[ob_index]
    cut = cuts[ob_index]
    
    x = []
    for i in range(0, len(binedges)-1):
        midpoint = 0.5 * (binedges[i] + binedges[i+1])
        x.append(midpoint)
    
    dataframe_cut = applyCut(dataframe, cut)
    eff = get_eff(ob, binedges, cut, file='/vast/palmer/home.grace/ys668/output/run12/all_' + str(detector_variation) + '.root')
    
    y_arr = []
    err_arr = []
    
    for it in range(2,3):
        print("iteration", it)
        y_seed, err_seed, err_stat = get_seed_unc(dataframe_cut, it, ob, binedges, seed_string="_" + str(detector_variation) + "seed")
        rel_err = np.sqrt( (err_stat/y_seed)**2 + (err_seed/y_seed)**2 )
        print('total (stat + NN):', rel_err)
        err = y_seed * rel_err
        
        # scale by efficiency
        y_scaled = y_seed / (1e-50 + eff)
        err_scaled = err / (1e-50 + eff)
        
        # normalize
        bin_widths = np.diff(binedges)
        y_arr.append(y_scaled / sum(y_scaled) / bin_widths)
        err_arr.append(err_scaled / sum(y_scaled) / bin_widths)
    
    return x, y_arr, err_arr


# $Q$

In [ ]:
obs = ['q','q','q','q','q']
binedgess = [np.arange(-1.0,1.2,0.2),np.arange(-1.0,1.2,0.2),np.arange(-1.0,1.2,0.2),np.arange(-1.0,1.2,0.2),np.arange(-0.8,1.2,0.4)]
cuts = ['20<pt<25', '25<pt<30', '30<pt<40', '40<pt<60', '40<pt<60']

In [ ]:
x, y_arr, err_arr = process_ob_nom(0)
# x, y_arr_hw, err_arr_hw = process_ob_hw(0)
# x, y_arr_py8, err_arr_py8 = process_ob_py8(0)
x, y_arr_eff096, err_arr_eff096 = process_ob_detector(0, "eff096", df1_eff096)
x, y_arr_tow104, err_arr_tow104 = process_ob_detector(0, "tow104", df1_tow104)
x, y_arr_hadcorr05, err_arr_hadcorr05 = process_ob_detector(0, "hadcorr05", df1_hadcorr05)

In [ ]:
err1 = abs(np.array(y_arr[1])-np.array(y_arr_eff096[0]))
err2 = abs(np.array(y_arr[1])-np.array(y_arr_tow104[0]))
err3 = abs(np.array(y_arr[1])-np.array(y_arr_hadcorr05[0]))
err4 = 0 # abs(np.array(y_arr[1])-np.array(y_arr_hw[0]))
err5 = 0 # abs(np.array(y_arr[1])-np.array(y_arr_py8[0]))
err6 = err_arr[1]

delta_y = []
for it in range(2,7): 
    delta_y.append( abs(y_arr[it-1]-y_arr[1]) )
err7 = np.max(delta_y,axis=0)

err = []
for i in range(0,len(err1)):
    err.append( np.sqrt( (err1[i])**2 + (err2[i])**2 + (err3[i])**2 + (max(abs(err4[i]), abs(err5[i]), abs(err7[i])))**2 + (err6[i])**2 ) )
err = np.array(err)

In [ ]:
plt.figure(figsize=(4,4))

plt.plot(x, (err/y_arr[1]), color='k', label='all')
plt.plot(x, (err3/y_arr[1]), '--', label='had. corr.')
plt.plot(x, (err2/y_arr[1]), '--', label='tower')
plt.plot(x, (err1/y_arr[1]), '--', label='tracking')
# plt.plot(x, (err4/y_arr[1]), '--', label='Herwig')
# plt.plot(x, (err5/y_arr[1]), '--', label='Pythia8')
plt.plot(x, (err7/y_arr[1]), '--', label='iteration')
plt.plot(x, (err6/y_arr[1]), '--', label='stochastic')

plt.ylabel('relative uncertainty')
plt.xlabel('$Q$')
plt.legend(frameon=False, loc='upper left', ncol=2)
plt.text(-0.5,0.22,'$p$+$p$ $\sqrt{s}=200$ GeV\nanti-$k_\mathrm{T}$, $R=0.4$')
plt.text(-0.5,0.18,'$20<p_T<25$ GeV/c')
plt.xlim(-1,1)
plt.ylim(0,0.6)

In [ ]:
mean_20_25 = y_arr[1]
err_20_25 = err